In [1]:
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Python:", sys.executable)
print("PySpark imported successfully.")

Python: C:\Users\HP\anaconda3\python.exe
PySpark imported successfully.


In [4]:
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

print("PYSPARK_PYTHON:", os.environ["PYSPARK_PYTHON"])
print("PYSPARK_DRIVER_PYTHON:", os.environ["PYSPARK_DRIVER_PYTHON"])

PYSPARK_PYTHON: C:\Users\HP\anaconda3\python.exe
PYSPARK_DRIVER_PYTHON: C:\Users\HP\anaconda3\python.exe


In [5]:
spark = (
    SparkSession.builder
    .appName("ChicagoCrime_EDA")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

Spark version: 4.2.0


In [8]:
BASE_PATH = r"C:\Users\HP\Documents\Big Data Technology"

RAW_PATH = os.path.join(BASE_PATH, "data", "raw")
PROCESSED_PATH = os.path.join(BASE_PATH, "data", "processed")

CRIME_PARQUET_PATH = os.path.join(
    PROCESSED_PATH,
    "integrated_crime_parquet"
)

print("Base path:", BASE_PATH)
print("Processed path:", PROCESSED_PATH)
print("Parquet path:", CRIME_PARQUET_PATH)

Base path: C:\Users\HP\Documents\Big Data Technology
Processed path: C:\Users\HP\Documents\Big Data Technology\data\processed
Parquet path: C:\Users\HP\Documents\Big Data Technology\data\processed\integrated_crime_parquet


In [9]:
#Check whether the integrated Parquet dataset exists
print("Parquet path exists:", os.path.exists(CRIME_PARQUET_PATH))

if os.path.exists(CRIME_PARQUET_PATH):
    print("Contents:")
    print(os.listdir(CRIME_PARQUET_PATH)[:10])

Parquet path exists: True
Contents:
['crime_year=2001', 'crime_year=2002', 'crime_year=2003', 'crime_year=2004', 'crime_year=2005', 'crime_year=2006', 'crime_year=2007', 'crime_year=2008', 'crime_year=2009', 'crime_year=2010']


In [10]:
#Load the integrated dataset
integrated_crime = spark.read.parquet(CRIME_PARQUET_PATH)

print("Integrated dataset loaded successfully.")
print("Rows:", integrated_crime.count())
print("Columns:", len(integrated_crime.columns))

Integrated dataset loaded successfully.
Rows: 7784664
Columns: 48


In [11]:
integrated_crime.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: double (nullable = true)
 |-- Y Coordinate: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)
 |-- crime_timestamp: timestamp (nullable = true)
 |-- crime_date: date (nullable = true)
 |-- crim

In [12]:
#Inspect the integrated dataset
integrated_crime.show(5, truncate=False)
print(integrated_crime.columns)

+-------+-----------+----------------------+---------------------+----+-------------+---------------------------+------------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+-----------------------------+-------------------+----------+-----------+---------+----------+-----------------+--------------------+--------------+---------------+-------------+--------+---------------+---------------+---------------+----------+----------+-------------------+-----------+----------------+------------------+-----------------+-----------------+--------------+-------------------+----------+--------------+
|ID     |Case Number|Date                  |Block                |IUCR|Primary Type |Description                |Location Description    |Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On            |Latitude    |Longitude    |Location             

In [13]:
#Check important columns
important_columns = [
    "crime_date",
    "crime_year",
    "crime_month",
    "crime_hour",
    "district_clean",
    "community_area_clean",
    "latitude_clean",
    "longitude_clean"
]

for col_name in important_columns:
    print(
        col_name,
        "->",
        col_name in integrated_crime.columns
    )

crime_date -> True
crime_year -> True
crime_month -> True
crime_hour -> True
district_clean -> True
community_area_clean -> True
latitude_clean -> True
longitude_clean -> True


In [14]:
#Create Spark SQL view
integrated_crime.createOrReplaceTempView("crime_data")

print("SQL view 'crime_data' created successfully.")

SQL view 'crime_data' created successfully.


In [15]:
#Test the SQL view
spark.sql("""
SELECT *
FROM crime_data
LIMIT 5
""").show(5, truncate=False)

+-------+-----------+----------------------+---------------------+----+-------------+---------------------------+------------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+-----------------------------+-------------------+----------+-----------+---------+----------+-----------------+--------------------+--------------+---------------+-------------+--------+---------------+---------------+---------------+----------+----------+-------------------+-----------+----------------+------------------+-----------------+-----------------+--------------+-------------------+----------+--------------+
|ID     |Case Number|Date                  |Block                |IUCR|Primary Type |Description                |Location Description    |Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On            |Latitude    |Longitude    |Location             

In [16]:
#Crime count by year
crime_by_year = spark.sql("""
SELECT
    crime_year,
    COUNT(*) AS crime_count
FROM crime_data
WHERE crime_year IS NOT NULL
GROUP BY crime_year
ORDER BY crime_year
""")

crime_by_year.show(30)

+----------+-----------+
|crime_year|crime_count|
+----------+-----------+
|      2001|     485878|
|      2002|     486802|
|      2003|     475979|
|      2004|     469421|
|      2005|     453771|
|      2006|     448174|
|      2007|     437084|
|      2008|     427167|
|      2009|     392819|
|      2010|     370496|
|      2011|     351964|
|      2012|     336262|
|      2013|     307468|
|      2014|     275731|
|      2015|     264755|
|      2016|     269786|
|      2017|     269071|
|      2018|     268773|
|      2019|     261245|
|      2020|     212092|
|      2021|     208571|
|      2022|     238215|
|      2023|      73140|
+----------+-----------+



In [17]:
#Crime count by month
crime_by_month = spark.sql("""
SELECT
    crime_month,
    COUNT(*) AS crime_count
FROM crime_data
WHERE crime_month IS NOT NULL
GROUP BY crime_month
ORDER BY crime_month
""")

crime_by_month.show(12)

+-----------+-----------+
|crime_month|crime_count|
+-----------+-----------+
|          1|     621870|
|          2|     547454|
|          3|     649788|
|          4|     641066|
|          5|     682819|
|          6|     681640|
|          7|     717118|
|          8|     710301|
|          9|     668097|
|         10|     676128|
|         11|     608818|
|         12|     579565|
+-----------+-----------+



In [18]:
#Crime count by hour
crime_by_hour = spark.sql("""
SELECT
    crime_hour,
    COUNT(*) AS crime_count
FROM crime_data
WHERE crime_hour IS NOT NULL
GROUP BY crime_hour
ORDER BY crime_hour
""")

crime_by_hour.show(24)

+----------+-----------+
|crime_hour|crime_count|
+----------+-----------+
|         0|     444272|
|         1|     246808|
|         2|     208146|
|         3|     168308|
|         4|     127941|
|         5|     106840|
|         6|     124613|
|         7|     177870|
|         8|     263432|
|         9|     336467|
|        10|     329969|
|        11|     345202|
|        12|     446899|
|        13|     369553|
|        14|     392468|
|        15|     414378|
|        16|     393014|
|        17|     399856|
|        18|     425556|
|        19|     438502|
|        20|     436963|
|        21|     423861|
|        22|     416908|
|        23|     346838|
+----------+-----------+



In [19]:
#Crime count by day of week
[c for c in integrated_crime.columns if "day" in c.lower()]

['crime_day', 'crime_day_of_week']

In [22]:
#Crime count by Primary Type

crime_types = spark.sql("""
SELECT
    `Primary Type` AS primary_type,
    COUNT(*) AS crime_count
FROM crime_data
WHERE `Primary Type` IS NOT NULL
GROUP BY `Primary Type`
ORDER BY crime_count DESC
LIMIT 15
""")

crime_types.show(15, truncate=False)

+--------------------------+-----------+
|primary_type              |crime_count|
+--------------------------+-----------+
|THEFT                     |1642148    |
|BATTERY                   |1422913    |
|CRIMINAL DAMAGE           |887266     |
|NARCOTICS                 |747633     |
|ASSAULT                   |507296     |
|OTHER OFFENSE             |483642     |
|BURGLARY                  |424397     |
|MOTOR VEHICLE THEFT       |375495     |
|DECEPTIVE PRACTICE        |344940     |
|ROBBERY                   |292334     |
|CRIMINAL TRESPASS         |214316     |
|WEAPONS VIOLATION         |106418     |
|PROSTITUTION              |69840      |
|OFFENSE INVOLVING CHILDREN|55719      |
|PUBLIC PEACE VIOLATION    |52325      |
+--------------------------+-----------+



In [23]:
#Spatial crime analysis by district
crime_by_district = spark.sql("""
SELECT
    district_clean,
    COUNT(*) AS crime_count
FROM crime_data
WHERE district_clean IS NOT NULL
GROUP BY district_clean
ORDER BY crime_count DESC
""")

crime_by_district.show(30)

+--------------+-----------+
|district_clean|crime_count|
+--------------+-----------+
|             8|     523193|
|            11|     499808|
|             6|     455120|
|             7|     450420|
|            25|     442880|
|             4|     441996|
|             3|     394841|
|            12|     385665|
|             9|     380367|
|             2|     368384|
|            18|     349314|
|            19|     349312|
|             5|     345021|
|            10|     335394|
|            15|     333750|
|             1|     314598|
|            14|     301290|
|            16|     260482|
|            22|     255159|
|            24|     235064|
|            17|     224697|
|            20|     137624|
|            31|        234|
|            21|          4|
+--------------+-----------+



In [24]:
#Spatial crime analysis by community area
crime_by_community = spark.sql("""
SELECT
    community_area_clean,
    COUNT(*) AS crime_count
FROM crime_data
WHERE community_area_clean IS NOT NULL
GROUP BY community_area_clean
ORDER BY crime_count DESC
LIMIT 20
""")

crime_by_community.show(20)

+--------------------+-----------+
|community_area_clean|crime_count|
+--------------------+-----------+
|                  25|     448276|
|                   8|     252839|
|                  43|     236555|
|                  23|     223982|
|                  28|     217006|
|                  24|     210238|
|                  29|     209901|
|                  67|     205117|
|                  71|     203100|
|                  49|     190600|
|                  68|     187126|
|                  69|     178267|
|                  32|     177732|
|                  66|     174517|
|                  44|     158031|
|                  22|     148347|
|                   6|     144756|
|                  61|     144352|
|                  26|     135522|
|                  27|     134276|
+--------------------+-----------+



In [25]:
#Crime rate / rolling average across community areas
monthly_community_crime = spark.sql("""
SELECT
    community_area_clean,
    crime_year,
    crime_month,
    COUNT(*) AS monthly_crime_count
FROM crime_data
WHERE community_area_clean IS NOT NULL
  AND crime_year IS NOT NULL
  AND crime_month IS NOT NULL
GROUP BY
    community_area_clean,
    crime_year,
    crime_month
""")

monthly_community_crime.createOrReplaceTempView("monthly_community_crime")

monthly_community_crime.show(10)

+--------------------+----------+-----------+-------------------+
|community_area_clean|crime_year|crime_month|monthly_crime_count|
+--------------------+----------+-----------+-------------------+
|                  56|      2005|         12|                289|
|                  66|      2005|         12|                857|
|                  65|      2005|          3|                285|
|                  66|      2005|          2|                729|
|                  56|      2005|          8|                348|
|                  70|      2005|          1|                315|
|                  56|      2005|          9|                324|
|                  58|      2005|          6|                390|
|                  58|      2005|          2|                389|
|                  65|      2005|          4|                274|
+--------------------+----------+-----------+-------------------+
only showing top 10 rows


In [26]:
#Calculate the 3-month rolling average
rolling_crime = spark.sql("""
SELECT
    community_area_clean,
    crime_year,
    crime_month,
    monthly_crime_count,

    AVG(monthly_crime_count) OVER (
        PARTITION BY community_area_clean
        ORDER BY crime_year, crime_month
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS rolling_3_month_avg

FROM monthly_community_crime
ORDER BY community_area_clean, crime_year, crime_month
""")

rolling_crime.show(20)

+--------------------+----------+-----------+-------------------+-------------------+
|community_area_clean|crime_year|crime_month|monthly_crime_count|rolling_3_month_avg|
+--------------------+----------+-----------+-------------------+-------------------+
|                   1|      2001|          1|                 11|               11.0|
|                   1|      2001|          2|                  3|                7.0|
|                   1|      2001|          3|                  4|                6.0|
|                   1|      2001|          4|                  5|                4.0|
|                   1|      2001|          5|                  3|                4.0|
|                   1|      2001|          6|                  4|                4.0|
|                   1|      2001|          7|                  3| 3.3333333333333335|
|                   1|      2001|          8|                  5|                4.0|
|                   1|      2001|          9|         

In [27]:
#Weather event correlation
weather_columns = [
    c for c in integrated_crime.columns
    if "weather" in c.lower()
    or "rain" in c.lower()
    or "snow" in c.lower()
    or "temperature" in c.lower()
    or "precip" in c.lower()
]

print(weather_columns)

['precipitation', 'snowfall', 'avg_temperature', 'max_temperature', 'min_temperature', 'rain_event', 'snow_event']


In [28]:
#Check weather event columns
event_columns = [
    c for c in integrated_crime.columns
    if "event" in c.lower()
]

print("Event-related columns:")
for c in event_columns:
    print(c)

Event-related columns:
rain_event
snow_event


In [29]:
#Crime frequency on rain-event days
rain_crime = spark.sql("""
SELECT
    rain_event,
    COUNT(*) AS crime_count
FROM crime_data
WHERE rain_event IS NOT NULL
GROUP BY rain_event
ORDER BY rain_event
""")

rain_crime.show()

+----------+-----------+
|rain_event|crime_count|
+----------+-----------+
|         0|    5242191|
|         1|    2540785|
+----------+-----------+



In [30]:
#Crime frequency on snow-event days
snow_crime = spark.sql("""
SELECT
    snow_event,
    COUNT(*) AS crime_count
FROM crime_data
WHERE snow_event IS NOT NULL
GROUP BY snow_event
ORDER BY snow_event
""")

snow_crime.show()

+----------+-----------+
|snow_event|crime_count|
+----------+-----------+
|         0|    7746957|
|         1|      36019|
+----------+-----------+



In [31]:
#Calculate crime rate per day for weather events
rain_daily = spark.sql("""
SELECT
    crime_date,
    rain_event,
    COUNT(*) AS daily_crime_count
FROM crime_data
WHERE crime_date IS NOT NULL
  AND rain_event IS NOT NULL
GROUP BY crime_date, rain_event
""")

rain_daily.groupBy("rain_event").agg(
    F.count("*").alias("number_of_days"),
    F.avg("daily_crime_count").alias("average_daily_crime")
).show()

+----------+--------------+-------------------+
|rain_event|number_of_days|average_daily_crime|
+----------+--------------+-------------------+
|         1|          2705|  939.2920517560074|
|         0|          5439|   963.815223386652|
+----------+--------------+-------------------+



In [32]:
#Weather correlation
daily_weather_crime = spark.sql("""
SELECT
    crime_date,
    COUNT(*) AS daily_crime_count,
    AVG(precipitation) AS precipitation
FROM crime_data
WHERE crime_date IS NOT NULL
  AND precipitation IS NOT NULL
GROUP BY crime_date
ORDER BY crime_date
""")

daily_weather_crime.show(20)

+----------+-----------------+-------------+
|crime_date|daily_crime_count|precipitation|
+----------+-----------------+-------------+
|2001-01-01|             1825|          0.0|
|2001-01-02|             1143|          0.0|
|2001-01-03|             1151|          0.0|
|2001-01-04|             1166|          0.0|
|2001-01-05|             1267|          0.0|
|2001-01-06|             1293|          0.0|
|2001-01-07|             1105|          0.0|
|2001-01-08|             1157|          0.0|
|2001-01-09|             1184|          0.0|
|2001-01-10|             1237|          0.0|
|2001-01-11|             1228|          0.0|
|2001-01-12|             1353|          0.0|
|2001-01-13|             1311|          0.0|
|2001-01-14|             1119|         58.0|
|2001-01-15|             1312|          3.0|
|2001-01-16|             1251|          0.0|
|2001-01-17|             1288|          0.0|
|2001-01-18|             1278|          0.0|
|2001-01-19|             1214|          0.0|
|2001-01-2

In [33]:
#Broadcast join demonstration
community_names = (
    integrated_crime
    .select(
        "community_area_clean",
        "community_area_name"
    )
    .where(F.col("community_area_clean").isNotNull())
    .dropDuplicates(["community_area_clean"])
)

print("Community lookup rows:", community_names.count())

community_names.show(20, truncate=False)

Community lookup rows: 77
+--------------------+-------------------+
|community_area_clean|community_area_name|
+--------------------+-------------------+
|1                   |Rogers Park        |
|2                   |West Ridge         |
|3                   |Uptown             |
|4                   |Lincoln Square     |
|5                   |North Center       |
|6                   |Lake View          |
|7                   |Lincoln Park       |
|8                   |Near North Side    |
|9                   |Edison Park        |
|10                  |Norwood Park       |
|11                  |Jefferson Park     |
|12                  |Forest Glen        |
|13                  |North Park         |
|14                  |Albany Park        |
|15                  |Portage Park       |
|16                  |Irving Park        |
|17                  |Dunning            |
|18                  |Montclaire         |
|19                  |Belmont Cragin     |
|20                  |Hermos

In [36]:
#Broadcast join

broadcast_joined = (
    integrated_crime.alias("crime")
    .join(
        F.broadcast(community_names).alias("community"),
        F.col("crime.community_area_clean")
        == F.col("community.community_area_clean"),
        "left"
    )
    .select(
        F.col("crime.community_area_clean").alias("community_area"),
        F.col("community.community_area_name"),
        F.col("crime.`Primary Type`").alias("primary_type"),
        F.col("crime.crime_date"),
        F.col("crime.district_clean")
    )
)

broadcast_joined.show(20, truncate=False)

+--------------+-------------------+-------------------+----------+--------------+
|community_area|community_area_name|primary_type       |crime_date|district_clean|
+--------------+-------------------+-------------------+----------+--------------+
|57            |Archer Heights     |THEFT              |2005-12-14|8             |
|66            |Chicago Lawn       |BATTERY            |2005-03-03|8             |
|56            |Garfield Ridge     |OTHER OFFENSE      |2005-12-29|8             |
|70            |Ashburn            |BATTERY            |2005-03-03|8             |
|65            |West Lawn          |THEFT              |2005-12-29|8             |
|66            |Chicago Lawn       |ROBBERY            |2005-03-02|8             |
|66            |Chicago Lawn       |MOTOR VEHICLE THEFT|2005-12-01|8             |
|64            |Clearing           |BURGLARY           |2005-03-02|8             |
|57            |Archer Heights     |MOTOR VEHICLE THEFT|2005-12-28|8             |
|70 

In [37]:
#Verify the Broadcast Hash Join
broadcast_joined.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (14)
+- Project (13)
   +- BroadcastHashJoin LeftOuter BuildRight (12)
      :- Project (2)
      :  +- Scan parquet  (1)
      +- BroadcastExchange (11)
         +- SortAggregate (10)
            +- Sort (9)
               +- Exchange (8)
                  +- SortAggregate (7)
                     +- Sort (6)
                        +- Project (5)
                           +- Filter (4)
                              +- Scan parquet  (3)


(1) Scan parquet 
Output [5]: [Primary Type#5, crime_date#23, community_area_clean#28, crime_year#46, district_clean#47]
Batched: true
Location: InMemoryFileIndex [file:/C:/Users/HP/Documents/Big Data Technology/data/processed/integrated_crime_parquet]
ReadSchema: struct<Primary Type:string,crime_date:date,community_area_clean:int>

(2) Project
Output [4]: [Primary Type#5, crime_date#23, community_area_clean#28, district_clean#47]
Input [5]: [Primary Type#5, crime_date#23, community_area_clean#28, crime_year#46,

In [38]:
#Create Task 2 summary table
district_daily = spark.sql("""
SELECT
    district_clean,
    crime_date,
    COUNT(*) AS daily_crimes
FROM crime_data
WHERE district_clean IS NOT NULL
  AND crime_date IS NOT NULL
GROUP BY district_clean, crime_date
""")

district_daily_summary = (
    district_daily
    .groupBy("district_clean")
    .agg(
        F.avg("daily_crimes").alias("average_daily_crimes"),
        F.max("daily_crimes").alias("maximum_daily_crimes"),
        F.count("*").alias("number_of_days")
    )
    .orderBy(F.desc("average_daily_crimes"))
)

district_daily_summary.show(30)

+--------------+--------------------+--------------------+--------------+
|district_clean|average_daily_crimes|maximum_daily_crimes|number_of_days|
+--------------+--------------------+--------------------+--------------+
|             8|    64.2269825681316|                 166|          8146|
|            11|   61.35624846550454|                 283|          8146|
|             6|  55.870365823717165|                 172|          8146|
|             7|   55.29339553154923|                 150|          8146|
|            25|    54.3677878713479|                 170|          8146|
|             4|   54.25926835256568|                 138|          8146|
|             3|   48.47053768720845|                 120|          8146|
|            12|  47.344095261478024|                 108|          8146|
|             9|   46.69371470660447|                 116|          8146|
|             2|    45.2226859808495|                 165|          8146|
|            18|   42.88165971028726| 

In [39]:
#EDA outputs
TASK2_PATH = os.path.join(
    PROCESSED_PATH,
    "task2_eda"
)

os.makedirs(TASK2_PATH, exist_ok=True)

print("Task 2 output folder:", TASK2_PATH)



Task 2 output folder: C:\Users\HP\Documents\Big Data Technology\data\processed\task2_eda


In [41]:
crime_by_year.coalesce(1).write.mode("overwrite").option(
    "header", "true"
).csv(os.path.join(TASK2_PATH, "crime_by_year"))

crime_by_month.coalesce(1).write.mode("overwrite").option(
    "header", "true"
).csv(os.path.join(TASK2_PATH, "crime_by_month"))

crime_by_hour.coalesce(1).write.mode("overwrite").option(
    "header", "true"
).csv(os.path.join(TASK2_PATH, "crime_by_hour"))

print("Task 2 summary datasets saved.")

Task 2 summary datasets saved.
